# Template 01: Data Assembly

**Purpose:** Load and merge master and auxiliary datasets

**Inputs:**
- config_path: Path to config YAML
- master_dataset_car.parquet
- car_with_dep_factor_fold.parquet

**Outputs:**
- data/01_assembled.parquet (merged data with fold column)

**Memory:** Load both files, join, filter to debug folds, then save

In [1]:
# Parameters (injected by papermill)
config_path = "config/car_coll/v1"

In [2]:
# Parameters
config_path = "config/car_coll/v1"


In [3]:
# Imports
import pandas as pd
import numpy as np
import yaml
import os
import sys
import gc
from datetime import datetime
from pathlib import Path

# Add lib to path and setup environment
sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment

print("########################################")
print("# STAGE 01: DATA ASSEMBLY")
print("########################################")

# Auto-detect project root (cross-machine compatible)
project_root = setup_notebook_environment()
print(f"\nProject root: {project_root}")

########################################
# STAGE 01: DATA ASSEMBLY
########################################

Project root: /Users/Mach/dev/aps/code/26Dmodelv1


In [4]:
# Load config (with machine-specific paths)
from utils import load_config
config_file = f"{config_path}/config.yaml"
print(f"\n* Loading config: {config_file}")

cfg = load_config(config_file)

print(f"  Experiment: {cfg['experiment']['name']}")
print(f"  Target: {cfg['experiment']['target']}")
print(f"  Debug level: {cfg['debug']['level']}")


* Loading config: config/car_coll/v1/config.yaml
  Experiment: car_bi_v1
  Target: pp_bi
  Debug level: 1


In [5]:
# Setup paths
master_data_path = cfg["paths"]["master_data_path"]
aux_data_path = cfg["paths"]["aux_data_path"]
output_base = cfg["paths"]["output_base"]
os.makedirs(f"{output_base}/data", exist_ok=True)

master_path = f"{master_data_path}/{cfg['data']['master_file']}"
aux_path = f"{aux_data_path}/{cfg['data']['aux_file']}"

print(f"\n* Data paths:")
print(f"  Master: {master_path}")
print(f"  Aux: {aux_path}")
print(f"  Output: {output_base}")


* Data paths:
  Master: /Users/Mach/dev/aps/data/2026_Dmodel_data/master_dataset_car.parquet
  Aux: /Users/Mach/dev/aps/data/2026_Dmodel_data/car_with_dep_factor_fold.parquet
  Output: output/car_coll/v1


In [6]:
# Determine which folds to load based on debug level
debug_level = cfg['debug']['level']
test_fold = cfg['debug']['test_fold']
train_folds = cfg['debug']['fold_mapping'][debug_level]

print(f"\n* Fold configuration:")
print(f"  Debug level: {debug_level}")
print(f"  Train folds: {train_folds}")
print(f"  Test fold: {test_fold}")


* Fold configuration:
  Debug level: 1
  Train folds: [1]
  Test fold: 6


In [7]:
# Load auxiliary file FIRST (smaller, has fold column)
print(f"\n* Loading auxiliary file...")
aux_data = pd.read_parquet(aux_path)
print(f"  Shape: {aux_data.shape}")
print(f"  Memory: {aux_data.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"  Columns: {aux_data.columns.tolist()[:10]}...")

# Check for fold column
fold_col = cfg['data']['fold_column']
if fold_col in aux_data.columns:
    print(f"  ! Fold column found: {fold_col}")
    print(f"  ! Fold values: {sorted(aux_data[fold_col].unique().tolist())}")
else:
    print(f"  ! ERROR: Fold column '{fold_col}' not found in aux data")
    print(f"  ! Available columns: {aux_data.columns.tolist()}")


* Loading auxiliary file...


  Shape: (27052200, 19)
  Memory: 5.77 GB
  Columns: ['vin_date', 'vin', 'BODY_STYLE_SEGMENT_BODY_TYPE', 'cef_est_curr_mi_grp_imps', 'ODOMETER_IMP_FLAG', 'geo_pop_density_ntile', 'POP_DENSITY_IMP_FLAG', 'CALC_VEH_AGE', 'st_raw', 'vc_msrp_impa']...
  ! Fold column found: fold
  ! Fold values: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [8]:
# Filter aux data to only needed folds (reduce memory before join)
needed_folds = train_folds + [test_fold]
aux_filtered = aux_data[aux_data[fold_col].isin(needed_folds)].copy()
print(f"\n* Filtered aux data to folds {needed_folds}")
print(f"  Shape: {aux_filtered.shape}")
print(f"  Memory saved: {(aux_data.memory_usage(deep=True).sum() - aux_filtered.memory_usage(deep=True).sum()) / 1e9:.2f} GB")

# Clean up original aux data
del aux_data
gc.collect()


* Filtered aux data to folds [1, 6]
  Shape: (5424070, 19)
  Memory saved: 4.57 GB


0

In [9]:
# Find join key in aux data
join_key_config = cfg['data']['join_key']
print(f"\n* Looking for join key '{join_key_config}'...")

# Try to find the join key (check variations)
possible_keys = [join_key_config, 'VIN_Date', 'vindate', 'vin_date', 'VINDATE']
join_key = None

for key in possible_keys:
    if key in aux_filtered.columns:
        join_key = key
        print(f"  ! Found join key: {join_key}")
        break

if join_key is None:
    print(f"  ! ERROR: Join key not found in aux data")
    print(f"  ! Aux columns: {aux_filtered.columns.tolist()[:20]}")


* Looking for join key 'vin_date'...
  ! Found join key: vin_date


In [10]:
# Load column subset (for performance)
columns_file = cfg['data'].get('columns_to_load_file')

if columns_file:
    columns_path = f"{config_path}/{columns_file}"
    if os.path.exists(columns_path):
        print(f"\n* Loading column subset from: {columns_file}")
        cols_df = pd.read_csv(columns_path, comment='#')
        columns_to_load = cols_df['column_name'].tolist()
        # Ensure join key is included
        if join_key not in columns_to_load:
            columns_to_load.append(join_key)
            print(f"  Added join key '{join_key}' to column list")
        print(f"  Columns requested: {len(columns_to_load)}")
    else:
        print(f"\n* WARNING: Column file not found: {columns_path}")
        print(f"  Loading ALL columns (slower)")
        print(f"  Run Template 00 (setup) first to create column subset file")
        columns_to_load = None
else:
    print(f"\n* No column subset specified, loading ALL columns")
    columns_to_load = None

# Load master file with column filtering
print(f"\n* Loading master file...")
if columns_to_load:
    # IMPORTANT: Filter to only columns that exist in master file
    # Target variables (pp_bi, etc.) are in AUX file, not master
    import pyarrow.parquet as pq
    master_schema = pq.read_schema(master_path)
    available_cols = set(master_schema.names)
    
    # Filter requested columns to only those available
    filtered_cols = [c for c in columns_to_load if c in available_cols]
    missing_cols = set(columns_to_load) - set(filtered_cols)
    
    if missing_cols:
        print(f"  ! Skipping {len(missing_cols)} columns not in master file:")
        for col in sorted(list(missing_cols)[:5]):
            print(f"    - {col}")
        if len(missing_cols) > 5:
            print(f"    ... and {len(missing_cols) - 5} more")
        print(f"    (These will be added from aux file during join)")
    
    print(f"  Loading {len(filtered_cols)} columns (optimized)")
    master_data = pd.read_parquet(master_path, columns=filtered_cols)
else:
    print(f"  Loading ALL columns (slower)")
    master_data = pd.read_parquet(master_path)

print(f"  Shape: {master_data.shape}")
print(f"  Memory: {master_data.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"  Columns (first 10): {master_data.columns.tolist()[:10]}")


* Loading column subset from: columns_to_load_during_dataassembly.csv
  Columns requested: 100

* Loading master file...
  ! Skipping 6 columns not in master file:
    - CALC_VEH_AGE
    - Dep_factor
    - fold
    - pp_bi
    - veh_value_dep
    ... and 1 more
    (These will be added from aux file during join)
  Loading 94 columns (optimized)


  Shape: (22705842, 94)


  Memory: 22.22 GB
  Columns (first 10): ['NumMajViol_raw', 'NumMajinAcc_raw', 'NumMinAcc_raw', 'NumMinViol_raw', 'NumSpdViol_raw', 'late_payments_raw', 'gender_male_ind', 'married_ind', 'multi_pol_unknown_cal', 'multi_pol_yes_cal']


In [11]:
# Check if join key exists in master
if join_key not in master_data.columns:
    print(f"\n* Join key '{join_key}' not in master data")
    print(f"* Trying alternative names...")
    
    for key in possible_keys:
        if key in master_data.columns:
            print(f"  ! Found '{key}' in master data")
            # Rename to match aux data
            master_data = master_data.rename(columns={key: join_key})
            print(f"  ! Renamed '{key}' -> '{join_key}'")
            break

In [12]:
# Perform join
print(f"\n* Joining master and aux data...")
print(f"  Join key: {join_key}")
print(f"  Join type: {cfg['data']['join_type']}")

data = master_data.merge(
    aux_filtered,
    on=join_key,
    how=cfg['data']['join_type']
)

print(f"  ! Joined shape: {data.shape}")
print(f"  ! Memory: {data.memory_usage(deep=True).sum() / 1e9:.2f} GB")

# Clean up
del master_data, aux_filtered
gc.collect()


* Joining master and aux data...
  Join key: vin_date
  Join type: inner


  ! Joined shape: (5424070, 112)


  ! Memory: 6.21 GB


0

In [13]:
# Verify fold column exists
if fold_col in data.columns:
    print(f"\n* Fold column verified in merged data")
    print(f"  Fold counts:")
    fold_counts = data[fold_col].value_counts().sort_index()
    for fold, count in fold_counts.items():
        if fold in train_folds:
            print(f"    Fold {fold}: {count:,} rows (TRAIN)")
        elif fold == test_fold:
            print(f"    Fold {fold}: {count:,} rows (TEST)")
        else:
            print(f"    Fold {fold}: {count:,} rows (other)")
else:
    print(f"  ! ERROR: Fold column missing after join!")


* Fold column verified in merged data
  Fold counts:
    Fold 1: 2,716,120 rows (TRAIN)
    Fold 6: 2,707,950 rows (TEST)


In [14]:
# Validate output before saving
from utils import validate_critical_columns
validate_critical_columns(data, cfg, 'Stage 01 Output')

# Save checkpoint
checkpoint_path = f"{output_base}/data/01_assembled.parquet"
print(f"\n* Saving checkpoint...")
data.to_parquet(checkpoint_path)

file_size = os.path.getsize(checkpoint_path) / 1e9
print(f"  ! Saved: {checkpoint_path}")
print(f"  ! Size: {file_size:.2f} GB")
print(f"  ! Shape: {data.shape}")

[Stage 01 Output] WARNING: pp_bi has 285,656 nulls (5.3%)
[Stage 01 Output] ✓ Critical columns validated: vin_date, fold, pp_bi, ee_bi_imps

* Saving checkpoint...


  ! Saved: output/car_coll/v1/data/01_assembled.parquet
  ! Size: 0.46 GB
  ! Shape: (5424070, 112)


In [15]:
# Memory cleanup
print(f"\n* Cleaning memory...")
del data
gc.collect()
print(f"  ! Memory cleaned")


* Cleaning memory...
  ! Memory cleaned


In [16]:
print("\n########################################")
print("# STAGE 01: COMPLETE")
print("########################################")


########################################
# STAGE 01: COMPLETE
########################################
